In [7]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

print("Модули импортированы")

C:\Users\Duck\AppData\Local\Temp\ipykernel_5716\2208943771.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


Модули импортированы


In [8]:
loader = DirectoryLoader(
    '../data/raw/kinopoisk_reviews',
    glob='**/*.txt',
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=True
)

print("Загружаем документы...")
documents = loader.load()
print(f"Загружено {len(documents)} документов")

print(f"\nПервый документ:")
print(f"   Content: {documents[0].page_content[:100]}...")
print(f"   Metadata: {documents[0].metadata}")

Загружаем документы...


100%|██████████| 131669/131669 [22:48<00:00, 96.23it/s] 

Загружено 131669 документов

Первый документ:
   Content: В 2003-ем году под руководством малоизвестного режиссёра Кларка Джонсона студия 'Columbia Pictures' ...
   Metadata: {'source': '..\\data\\raw\\kinopoisk_reviews\\neg\\1000083-0.txt'}


In [9]:
for doc in documents:
    source_path = doc.metadata['source']
    category = Path(source_path).parent.name
    doc.metadata['category'] = category

print("Распределение по категориям:")
from collections import Counter
categories = [doc.metadata['category'] for doc in documents]
print(Counter(categories))

print(f"\nПример документа:")
print(f"   Content: {documents[0].page_content[:100]}...")
print(f"   Metadata: {documents[0].metadata}")

Распределение по категориям:
Counter({'pos': 87138, 'neu': 24704, 'neg': 19827})

Пример документа:
   Content: В 2003-ем году под руководством малоизвестного режиссёра Кларка Джонсона студия 'Columbia Pictures' ...
   Metadata: {'source': '..\\data\\raw\\kinopoisk_reviews\\neg\\1000083-0.txt', 'category': 'neg'}


In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f"Было документов: {len(documents)}")
print(f"Стало чанков: {len(chunks)}")

for i, chunk in enumerate(chunks[:3]):
    print(f"\nЧанк {i+1}:")
    print(f"   Content: {chunk.page_content[:100]}...")
    print(f"   Metadata: {chunk.metadata}")

Было документов: 131669
Стало чанков: 878726

Чанк 1:
   Content: В 2003-ем году под руководством малоизвестного режиссёра Кларка Джонсона студия 'Columbia Pictures' ...
   Metadata: {'source': '..\\data\\raw\\kinopoisk_reviews\\neg\\1000083-0.txt', 'category': 'neg'}

Чанк 2:
   Content: посмотрело его небольшое количество зрителей (что объясняется не только низким бюджетом, но и качест...
   Metadata: {'source': '..\\data\\raw\\kinopoisk_reviews\\neg\\1000083-0.txt', 'category': 'neg'}

Чанк 3:
   Content: Действие начинается с того, как командиру Трэвису Холлу и его группе приходит задание сопроводить чр...
   Metadata: {'source': '..\\data\\raw\\kinopoisk_reviews\\neg\\1000083-0.txt', 'category': 'neg'}


In [11]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name='cointegrated/rubert-tiny2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': False}
)

test_vector = embeddings.embed_query("отличный фильм")
print(f"Размер вектора: {len(test_vector)}")
print(f"Первые 5 чисел: {test_vector[:5]}")

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Размер вектора: 312
Первые 5 чисел: [-0.05532436817884445, 0.02439088188111782, -0.05662994086742401, -0.058827679604291916, 0.026790881529450417]


In [12]:
import shutil
from pathlib import Path

CHROMA_PATH = 'data/chroma_langchain'

if Path(CHROMA_PATH).exists():
    shutil.rmtree(CHROMA_PATH)
    print(f"Старая папка {CHROMA_PATH} удалена")
else:
    print(f"Папка {CHROMA_PATH} не найдена (это нормально)")

Папка data/chroma_langchain не найдена (это нормально)


In [13]:
from langchain_chroma import Chroma

CHROMA_PATH = '../data/chroma_langchain'

vectorstore = Chroma(
    collection_name='kinopoisk_reviews_lc',
    embedding_function=embeddings,
    persist_directory=CHROMA_PATH
)

print(f"Chroma через LangChain создан")
print(f"Данные в: {CHROMA_PATH}")
print(f"Размер: {vectorstore._collection.count()} документов")

Chroma через LangChain создан
Данные в: ../data/chroma_langchain
Размер: 878726 документов


In [ ]:
#print("Добавляем чанки в ChromaDB...")
#
#BATCH_SIZE = 500
#for i in range(0, len(chunks), BATCH_SIZE):
#    batch = chunks[i:i+BATCH_SIZE]
#    vectorstore.add_documents(batch)
#    print(f"  Добавлено {min(i+BATCH_SIZE, len(chunks))} из {len(chunks)}")
#
#print(f"\nВсего в коллекции: {vectorstore._collection.count()} чанков")

Добавляем чанки в ChromaDB...


NameError: name 'vectorstore' is not defined

In [14]:
query = "отличный фильм с захватывающим сюжетом"

results = vectorstore.similarity_search(query, k=5)

print(f"Запрос: '{query}'\n")
for i, doc in enumerate(results, 1):
    print(f"{i}. [Категория: {doc.metadata['category']}]")
    print(f"   {doc.page_content[:150]}...")
    print()

Запрос: 'отличный фильм с захватывающим сюжетом'

1. [Категория: pos]
   Великолепный фильм с захватывающим сюжетом и неожиданной концовкой....

2. [Категория: neu]
   Удивительный фильм в своем жанре. 

Сюжет....

3. [Категория: pos]
   Потрясающий фильм....

4. [Категория: neu]
   очень простой фильм, с слишком предсказуемым сюжетом....

5. [Категория: pos]
   Отличный фильм....



In [15]:
results_with_scores = vectorstore.similarity_search_with_score(query, k=5)

for doc, distance in results_with_scores:
    similarity = 1 - distance  # для cosine метрики
    print(f"[Сходство: {similarity:.3f}] {doc.page_content[:80]}...")

[Сходство: 0.792] Великолепный фильм с захватывающим сюжетом и неожиданной концовкой....
[Сходство: 0.733] Удивительный фильм в своем жанре. 

Сюжет....
[Сходство: 0.707] Потрясающий фильм....
[Сходство: 0.691] очень простой фильм, с слишком предсказуемым сюжетом....
[Сходство: 0.684] Отличный фильм....


In [17]:
results_filtered = vectorstore.similarity_search(
    query,
    k=5,
    filter={'category': 'pos'}
)

print(f"Запрос: '{query}'")
print(f"Фильтр: category = 'pos'\n")

for i, doc in enumerate(results_filtered, 1):
    print(f"{i}. [{doc.metadata['category']}] {doc.page_content[:100]}...")

Запрос: 'отличный фильм с захватывающим сюжетом'
Фильтр: category = 'pos'

1. [pos] Великолепный фильм с захватывающим сюжетом и неожиданной концовкой....
2. [pos] Потрясающий фильм....
3. [pos] Отличный фильм....
4. [pos] Очень красивый мелодраматический фильм....
5. [pos] Захватывающий сюжет...


In [ ]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={
        'k': 5,
        'filter': {'category': 'pos'}
    }
)

print(f"Retriever создан")
print(f"Тип: {type(retriever)}")

query = "шедевр кинематографа"
docs = retriever.invoke(query)

print(f"\nЗапрос: '{query}'")
print(f"Найдено документов: {len(docs)}")
for i, doc in enumerate(docs, 1):
    print(f"\n{i}. [{doc.metadata['category']}]")
    print(f"   {doc.page_content[:120]}...")

Retriever создан
Тип: <class 'langchain_core.vectorstores.base.VectorStoreRetriever'>

Запрос: 'шедевр кинематографа'
Найдено документов: 5

1. [pos]
   Фильм - классика....

2. [pos]
   Киноклассика...

3. [pos]
   Жуткое кино....

4. [pos]
   Фильм, как акт искусства....

5. [pos]
   Душевное кино....


In [20]:
from langchain_core.prompts import PromptTemplate

rag_prompt = PromptTemplate(
    template="""Ты — ассистент по отзывам Кинопоиска. 
Ответь на вопрос пользователя, используя ТОЛЬКО информацию из контекста ниже.
Если в контексте нет ответа, скажи "Не знаю".

Контекст:
{context}

Вопрос: {question}

Ответ:""",
    input_variables=['context', 'question']
)

context = "\n\n".join([doc.page_content for doc in docs])
formatted_prompt = rag_prompt.format(context=context, question=query)

print("Сформированный промпт:\n")
print(formatted_prompt)

Сформированный промпт:

Ты — ассистент по отзывам Кинопоиска. 
Ответь на вопрос пользователя, используя ТОЛЬКО информацию из контекста ниже.
Если в контексте нет ответа, скажи "Не знаю".

Контекст:
Фильм - классика.

Киноклассика

Жуткое кино.

Фильм, как акт искусства.

Душевное кино.

Вопрос: шедевр кинематографа

Ответ:


In [24]:
from langchain_core.runnables import RunnablePassthrough 
from langchain_core.output_parsers import StrOutputParser  

def format_docs(docs):
    return "\n\n---\n\n".join([doc.page_content for doc in docs])

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
)

result = rag_chain.invoke(query)

print("Запрос:", query)
print("\nРезультат (промпт для LLM):")
print(result)

Запрос: шедевр кинематографа

Результат (промпт для LLM):
text='Ты — ассистент по отзывам Кинопоиска. \nОтветь на вопрос пользователя, используя ТОЛЬКО информацию из контекста ниже.\nЕсли в контексте нет ответа, скажи "Не знаю".\n\nКонтекст:\nФильм - классика.\n\n---\n\nКиноклассика\n\n---\n\nЖуткое кино.\n\n---\n\nФильм, как акт искусства.\n\n---\n\nДушевное кино.\n\nВопрос: шедевр кинематографа\n\nОтвет:'


In [25]:
test_queries = [
    "Что говорят про захватывающий сюжет?",
    "Есть ли отзывы про плохую актёрскую игру?",
    "Какие фильмы называют шедеврами?"
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Запрос: '{query}'")
    print('='*60)
    
    docs = retriever.invoke(query)
    context = format_docs(docs)
    
    prompt = rag_prompt.format(context=context, question=query)
    
    print(f"\nНайдено {len(docs)} документов:")
    for i, doc in enumerate(docs, 1):
        print(f"  {i}. [{doc.metadata['category']}] {doc.page_content[:80]}...")
    
    print(f"\nПромпт (первые 500 символов):")
    print(prompt[:500] + "...")


Запрос: 'Что говорят про захватывающий сюжет?'

Найдено 5 документов:
  1. [pos] А сюжет-то, собственно, в чём?...
  2. [pos] Что следует сказать про актерский состав?...
  3. [pos] Что можно сказать про документальный фильм?...
  4. [pos] Захватывающий сюжет...
  5. [pos] О построении сюжета...

Промпт (первые 500 символов):
Ты — ассистент по отзывам Кинопоиска. 
Ответь на вопрос пользователя, используя ТОЛЬКО информацию из контекста ниже.
Если в контексте нет ответа, скажи "Не знаю".

Контекст:
А сюжет-то, собственно, в чём?

---

Что следует сказать про актерский состав?

---

Что можно сказать про документальный фильм?

---

Захватывающий сюжет

---

О построении сюжета

Вопрос: Что говорят про захватывающий сюжет?

Ответ:...

Запрос: 'Есть ли отзывы про плохую актёрскую игру?'

Найдено 5 документов:
  1. [pos] Что следует сказать про актерский состав?...
  2. [pos] раз. Но, так ли плох фильм, как о нём принято говорить?...
  3. [pos] играют разные актеры. Чем не доказательство ге